# 18 · Feature engineering (baseline features)

## 1. Introduction

**What this notebook does:** it turns the cleaned listings into a **feature matrix** that a model can use, following
the plan of notebook 17 and the decisions taken after it. It builds the **baseline** feature set only.

**What this notebook does NOT do:** it does not train, tune or evaluate a model (that is notebook 19). It does not
change `cleaned_listings.csv` or `train_test_split.csv`, and it saves no files.

**The target** is the **advertised asking price** (`price`), not the actual sale price.

### Two kinds of transformation

This distinction is used throughout the notebook:

| Kind | What it means | Example | Leakage risk |
|---|---|---|---|
| **Fixed rule** | uses only the listing's **own** values and rules written down in advance | `"Ground"` → floor 0; balconies `"5"` → band `"4+"` | none: every listing gets the same answer whichever set it is in |
| **Learned transformation** | computes something from **many** listings (a median, a list of categories, a frequency threshold) | median bathrooms; which localities are "rare" | **yes**, if test listings take part. So these are **fitted on the training set only** |

The fixed rules are plain pandas functions stored in **`src/features/build_features.py`**, so that notebook 19 can
import exactly the same rules instead of copying them. This notebook imports them, explains each one (sections 7–11)
and shows what it does on training examples. To read the code of a function, open that file; each function has a
short comment explaining it. The functions are applied to train and test separately.
The learned transformations are collected in one scikit-learn `ColumnTransformer` (section 12), which is fitted on
the training set only (section 13) and then applied to both sets (section 14).

### Decisions this notebook follows

| # | Decision | Where |
|---|---|---|
| 1 | Rare localities: fewer than **10 training listings** → one shared "rare" category, learned from training data | 7, 12 |
| 2 | Apply the **safe free-text locality recovery** (the level-2 rule from notebook 10; candidates stored by notebook 15) | 7 |
| 3 | Include `prime_location` (the website's listing attribute, as it is) | 9 |
| 4 | `in_named_project` = 1 if the listing has a project ID (`raw_json.psmid`, stored as `project_id` by notebook 15) | 9 |
| 5 | No `total_floors` for houses/villas; floor information for apartment-like types only | 8 |
| 6 | Missing area: `area_missing` indicator + median imputation learned from training data | 8, 10 |
| 7 | Baseline features only; optional groups are tested later | 6 |
| 8 | `transaction_type` in three groups | 9 |
| 9 | Balconies in bands `1`, `2`, `3`, `4+`, `unknown` | 9 |
| 10 | Bathrooms: training-median imputation | 8, 10 |
| 11 | Also in the baseline (confirmed after notebook 17): `age_of_property`, `facing`, `posted_by`, `area_unit_is_sqyd`, `has_land_area_unit` | 8, 9 |
| 12 | Recovered localities get their centre from the website's locality table (`localities.csv`, read only) | 11 |
| 13 | Locality centres outside the Hyderabad box of notebook 14 → missing (the locality ID is kept) | 11 |
| 14 | No scaling here; notebook 19 adds a scaler inside the model pipeline if it uses a linear model | 12 |

## 2. Load the cleaned data and the train/test split

We read the three input files and take an MD5 fingerprint of each. At the end of the notebook the fingerprints are
checked again, to prove nothing was changed.

`localities.csv` is the website's locality table. We only use four of its columns: the locality ID, its name (to show
readable examples) and its centre coordinates (section 11). **No** price statistics are read from it.

In [1]:
from pathlib import Path
import hashlib
import sys

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

CLEANED_CSV = PROJECT_ROOT / "data" / "processed" / "cleaned_listings.csv"
SPLIT_CSV = PROJECT_ROOT / "data" / "processed" / "train_test_split.csv"
LOCALITIES_CSV = PROJECT_ROOT / "data" / "extracted" / "localities.csv"

# The fixed-rule functions live in src/features/build_features.py.
# Adding the project folder to Python's search path lets us import them from this notebook.
sys.path.insert(0, str(PROJECT_ROOT))
from src.features.build_features import (
    INPUT_COLUMNS,
    add_fixed_features,
    apply_safe_recovery,
    make_area_features,
    make_area_meaning_flags,
    make_balconies_band,
    make_binary_flags,
    make_centre_lookups,
    make_facing_clean,
    make_floor_features,
    make_locality_centre,
    make_locality_code,
    make_transaction_group,
    possession_to_status,
    rooms_to_number,
)


def file_md5(path):
    # A fingerprint of the file's bytes. If the file changes, the fingerprint changes.
    return hashlib.md5(path.read_bytes()).hexdigest()


md5_at_start = {}
for path in [CLEANED_CSV, SPLIT_CSV, LOCALITIES_CSV]:
    md5_at_start[path] = file_md5(path)
    print(f"{path.name:28} {md5_at_start[path]}")

cleaned_listings.csv         3a81a385a9aca28d5b70322e1e143c08
train_test_split.csv         10693e86d1d8a55e606399d718a8f556
localities.csv               8d9bb54dfc2fb49673e74b976ab73ace


In [2]:
listings = pd.read_csv(CLEANED_CSV, low_memory=False)
split_assignment = pd.read_csv(SPLIT_CSV)

# Only these columns of the locality table are read: ID, name and centre. No price statistics.
locality_table = pd.read_csv(LOCALITIES_CSV, usecols=["locality_id", "locality_name", "locality_lat", "locality_long"])

listings = listings.merge(split_assignment[["property_id", "split"]], on="property_id", validate="one_to_one")
print("Listings:", listings.shape)
print("Locality table:", locality_table.shape)

Listings: (20300, 77)
Locality table: (2009, 4)


## 3. Verify the split

Before building anything, we check that the split is the one notebook 16 created: the same row counts, every listing
assigned exactly once, and **no `property_group_id` on both sides**. (Notebook 16 used `GroupShuffleSplit` with
`test_size=0.20` and `random_state=42`; here we check its saved result.)

In [3]:
assert len(listings) == 20300
assert listings["property_id"].is_unique
assert listings["split"].isin(["train", "test"]).all()

train_groups = set(listings.loc[listings["split"] == "train", "property_group_id"])
test_groups = set(listings.loc[listings["split"] == "test", "property_group_id"])
assert train_groups.isdisjoint(test_groups), "A property group is on both sides!"

print(listings["split"].value_counts().to_string())
print("Property groups on both sides:", len(train_groups & test_groups))

split
train    16219
test      4081
Property groups on both sides: 0


## 4. Separate train and test

From here on, **every decision and every learned value comes from `train`**. The `test` table only receives
transformations: the same fixed rules, and the preprocessing learned on `train`. We never look at test values to
decide anything.

In [4]:
train = listings[listings["split"] == "train"].copy()
test = listings[listings["split"] == "test"].copy()

assert len(train) == 16219 and len(test) == 4081
print("Training rows:", len(train))
print("Test rows:    ", len(test))

del listings   # the combined table is no longer needed; this prevents using it by mistake

Training rows: 16219
Test rows:     4081


## 5. Define the target

`y = price`, the **advertised asking price** in rupees. It is kept in its own variables and never enters the
feature table.

Whether the model predicts `price` or `log(price)` is a **modelling** decision for notebook 19 (notebook 04 suggested a
log transform because prices are very right-skewed). It does not affect the features.

In [5]:
y_train = train["price"].copy()
y_test = test["price"].copy()

assert y_train.notna().all() and y_test.notna().all()
print("y_train:", len(y_train), " median asking price: ₹", round(y_train.median() / 1e5, 1), "lakh")
print("y_test: ", len(y_test))

y_train: 16219  median asking price: ₹ 150.0 lakh
y_test:  4081


## 6. Define the columns

**What:** `INPUT_COLUMNS` (defined in `build_features.py`) names exactly which **input** columns the fixed rules may
read. Here we check that none of them is forbidden.

**Why:** listing the allowed columns (instead of dropping forbidden ones) means a leaky column can never slip into the
features by accident.

**Leakage risk:** this section is the protection. The forbidden list below comes from the spec (section 4) and
notebook 17 (section 2).

**Baseline versus optional.** Only the baseline is built here. The optional groups are **not** read at all:
amenities, flooring, additional rooms, locality ratings, PIN code, area per bedroom, has-photo.

In [6]:
# INPUT_COLUMNS (imported): the columns the fixed rules are allowed to READ
print("Input columns read by the fixed rules:")
print(INPUT_COLUMNS)
print()

FORBIDDEN_COLUMNS = [
    "price", "price_per_sqft", "price_label", "price_band", "description", "raw_json",
    "loc_avg_price_sqft", "loc_avg_price_lmt", "loc_price_quarter_prev", "loc_price_quarter_current",
    "loc_growth_quarterly_pct", "loc_locality_rank", "scraped_at", "loc_scraped_at", "rowid",
    "listing_date", "posted_label", "detail_url",
    "property_id", "property_group_id", "advertiser_id", "owner_name", "pmt_source", "split",
]

OPTIONAL_LATER = [
    "amenities", "luxury_amenities", "flooring_type", "additional_rooms", "loc_zip_code", "image_url",
] + [column for column in train.columns if column.startswith("loc_rating_")]

assert set(INPUT_COLUMNS).isdisjoint(FORBIDDEN_COLUMNS), "A forbidden column is in the input list!"
assert set(INPUT_COLUMNS).isdisjoint(OPTIONAL_LATER)
missing_inputs = [column for column in INPUT_COLUMNS if column not in train.columns]
assert missing_inputs == [], missing_inputs

print(len(INPUT_COLUMNS), "input columns, none of them forbidden or optional.")

Input columns read by the fixed rules:
['property_type', 'bedrooms', 'bathrooms', 'balconies', 'furnishing', 'possession', 'age_of_property', 'facing', 'posted_by', 'transaction_type', 'prime_location', 'covered_area_sqft_converted', 'covered_area_unit', 'land_area_unit', 'floor', 'total_floors', 'locality_id', 'recovered_locality_id_candidate', 'loc_locality_lat', 'loc_locality_long', 'project_id']

21 input columns, none of them forbidden or optional.


**About `recovered_locality_id_candidate` and `project_id`:** notebook 15 labelled both "never a feature". That is
still true: neither is used **as a value**. The first is only used to fill in `locality_id` by the safe rule
(section 7), and the second only to create the yes/no flag `in_named_project` (section 9).

## 7. Locality handling

### 7.1 Safe free-text recovery (fixed rule)

**What:** some advertisers typed their locality as free text instead of choosing it from the website's list, so the
listing has no `locality_id`. Notebook 10 found a **safe** rule for some of them. After the "level 2" normalisation
(lower case, spaces collapsed, a 6-digit PIN code removed, a trailing "Hyderabad/Secunderabad/Telangana" removed), the text
is **exactly** the name of **one** locality ID. Notebook 15 stored that ID in `recovered_locality_id_candidate`
without applying it. We now apply it: if `locality_id` is missing and a candidate exists, the candidate becomes the
`locality_id`.

**Why:** these listings have a real, unambiguous locality. Leaving them as "unknown" would throw that information away.

**Fixed rule or learned?** Fixed. Each listing is handled using only its own text. No fuzzy matching, no substring
matching, no coordinates, and no "nearest locality" guess. The list of known names in notebook 15 contains locality
**names only** (no prices), so it cannot leak the target.

In [7]:
# apply_safe_recovery() is defined in src/features/build_features.py
train_locality_id = apply_safe_recovery(train)

recovered = train["locality_id"].isna() & train_locality_id.notna()
print("Training listings without a locality_id before:", train["locality_id"].isna().sum())
print("Recovered by the safe rule:                    ", recovered.sum())
print("Still without a locality_id:                   ", train_locality_id.isna().sum())

Training listings without a locality_id before: 1175
Recovered by the safe rule:                     85
Still without a locality_id:                    1090


In [8]:
# Example: what the rule did for three training listings
name_by_id = dict(zip(locality_table["locality_id"], locality_table["locality_name"]))

example = train.loc[recovered, ["locality", "locality_id"]].head(3).copy()
example["locality_id after recovery"] = train_locality_id[example.index]
example["name of that ID"] = example["locality_id after recovery"].map(name_by_id)
example

,locality,locality_id,locality_id after recovery,name of that ID
135,"Korremula, Hyderabad",NaN,93716.0,Korremula
392,"Neredmet, Hyderabad",NaN,85804.0,Neredmet
1031,Bacharam,NaN,113088.0,Bacharam


### 7.2 Locality as a category (fixed rule)

**What:** the locality ID is a code, not a quantity. Locality 84461 is not "bigger" than locality 50515, so it must
be treated as a **category**, not a number. We turn it into text (`84461.0` → `"84461"`). Every listing without an ID
gets the category `"unknown locality"`.

**Why "unknown locality" instead of dropping the row:** the investigation (`docs/locality_handling.md`) showed that
listings without a locality are systematically different (mostly owner adverts, cheaper). Dropping them would bias the
data, and guessing a locality would invent information.

In [9]:
# make_locality_code() is defined in src/features/build_features.py
train_locality_code = make_locality_code(train_locality_id)
print(train_locality_code.value_counts().head(6).to_string())

unknown locality    1090
84461                468
84654                371
84931                352
89377                351
87659                312


### 7.3 Rare localities (LEARNED from training data)

**What:** a locality with only a few training listings gives the model almost nothing to learn from. Localities with
**fewer than 10 training listings** are therefore grouped into one shared **rare** category.

**Why it must be learned from training data only:** "how many listings does this locality have?" is a count over many
listings. If test listings were included, a locality with 6 training and 5 test listings would count as "frequent"
only because of the test set.

**How it is implemented:** scikit-learn's `OneHotEncoder(min_frequency=10, handle_unknown="infrequent_if_exist")`
(section 12). When it is **fitted on the training data**, it counts each locality and remembers which ones have fewer
than 10 listings. When it **transforms** data:
- a frequent locality gets its own column (e.g. `locality_code_84461` = 1);
- a rare locality gets a 1 in the shared column `locality_code_infrequent_sklearn` (the "rare" column);
- a locality the encoder **never saw in training** (possible in the test set) also goes to that rare column;
- `"unknown locality"` is a normal category with more than 1,000 training listings, so it gets its own column.

Below, the same rule is written in plain pandas so you can see exactly what happens. The label `__RARE__` is only for
this illustration; in section 16 we check that the encoder learned exactly the same rare list.

In [10]:
MIN_LOCALITY_LISTINGS = 10

# LEARN: count listings per locality in the TRAINING data only
listings_per_locality = train_locality_code.value_counts()
frequent_localities = listings_per_locality[listings_per_locality >= MIN_LOCALITY_LISTINGS].index
rare_localities = listings_per_locality[listings_per_locality < MIN_LOCALITY_LISTINGS].index

print("Distinct locality categories in train:", len(listings_per_locality))
print("  frequent (>= 10 listings, own column):", len(frequent_localities))
print("  rare (< 10 listings, shared column):  ", len(rare_localities))
print()

# APPLY (illustration only): keep frequent localities, replace rare ones by "__RARE__"
locality_for_illustration = train_locality_code.where(train_locality_code.isin(frequent_localities), "__RARE__")

is_rare = locality_for_illustration == "__RARE__"
is_unknown = locality_for_illustration == "unknown locality"
print("Training listings with a frequent locality:", (~is_rare & ~is_unknown).sum())
print("Training listings with a rare locality:    ", is_rare.sum())
print("Training listings with unknown locality:   ", is_unknown.sum())

Distinct locality categories in train: 1835
  frequent (>= 10 listings, own column): 230
  rare (< 10 listings, shared column):   1605

Training listings with a frequent locality: 11266
Training listings with a rare locality:     3863
Training listings with unknown locality:    1090


In [11]:
# Examples: one frequent and one rare locality, before and after
# The most common real locality (skipping "unknown locality") and the first rare one
real_locality_counts = listings_per_locality.drop("unknown locality")
most_common_id = real_locality_counts.index[0]
rare_id = rare_localities[0]

for locality_code in [most_common_id, rare_id]:
    print(f"Before: locality = {name_by_id.get(int(locality_code), '?')!r} (ID {locality_code}), "
          f"{listings_per_locality[locality_code]} training listings")
    print(f"After:  locality = {locality_for_illustration[train_locality_code == locality_code].iloc[0]!r}")
    print()

Before: locality = 'Kondapur' (ID 84461), 468 training listings
After:  locality = '84461'

Before: locality = 'Bongloor' (ID 93664), 9 training listings
After:  locality = '__RARE__'



## 8. Numeric feature engineering (fixed rules)

### 8.1 Bedrooms (BHK) and bathrooms

**What:** both are stored as text because the website's last option is `"> 10"`. We convert them to numbers, with
`"> 10"` → **11**, meaning "11 or more". No listing has exactly 11, so 11 is not ambiguous (notebook 17, section 3.2).

**Fixed or learned?** Fixed. The 0.5% missing bathrooms are **not** filled here. That is a learned step (section 10).

In [12]:
# rooms_to_number() is defined in src/features/build_features.py
bedrooms_example = rooms_to_number(train["bedrooms"])
assert bedrooms_example.isna().sum() == train["bedrooms"].isna().sum(), "some text was not converted"

print(pd.DataFrame({"before (text)": train["bedrooms"], "after (number)": bedrooms_example})
      .drop_duplicates().sort_values("after (number)").to_string(index=False))

before (text)  after (number)
            1               1
            2               2
            3               3
            4               4
            5               5
            6               6
            7               7
            8               8
            9               9
           10              10
         > 10              11


### 8.2 Area: `log_area` and `area_missing`

**What:**
- `log_area = log1p(covered_area_sqft_converted)`, where `log1p(x)` = log(1 + x). The area was already converted to sq-ft
  with the **true** unit in notebook 15, and unreliable areas (rule R2) were already set to missing.
- `area_missing` = 1 when the area is missing, else 0.

**Why the log:** areas are very right-skewed (a few huge houses). On the log scale, doubling the area is the same step
whether a home goes from 800 to 1,600 or from 4,000 to 8,000 sq-ft, which matches how prices behave. (`log1p` rather than
`log` only guards against a zero; the smallest area here is 200 sq-ft, so the two are practically identical.)

**Why `area_missing`:** we **never invent** an area. In section 10 the missing `log_area` values are filled with a
training median, because most models cannot handle an empty cell. Without the indicator, the model could not tell a
**real** median-sized home from a home whose area is **unknown**. The indicator keeps that information.

**Fixed or learned?** Both columns are fixed rules. Only the fill value (section 10) is learned.

**Leakage risk:** none. `price_per_sqft` is not used anywhere.

In [13]:
# make_area_features() is defined in src/features/build_features.py
train_log_area, train_area_missing = make_area_features(train)

print("area_missing = 1 for", train_area_missing.sum(), "training listings",
      f"({train_area_missing.mean():.1%})")
print()
example_rows = [train.index[train["covered_area_sqft_converted"] == 1200][0],
                train.index[train["covered_area_sqft_converted"] == 2400][0],
                train.index[train["covered_area_sqft_converted"].isna()][0]]
pd.DataFrame({
    "covered_area_sqft_converted": train.loc[example_rows, "covered_area_sqft_converted"],
    "log_area": train_log_area[example_rows].round(3),
    "area_missing": train_area_missing[example_rows],
})

area_missing = 1 for 1390 training listings (8.6%)



,covered_area_sqft_converted,log_area,area_missing
36,1200.0,7.091,0
2996,2400.0,7.784,0
9,NaN,NaN,1


Notice: 2,400 sq-ft is twice 1,200 sq-ft, and on the log scale it is simply **+0.693** (= log 2) higher. The missing
area stays missing (NaN) for now. It is filled in section 10, and `area_missing` = 1 records that it was unknown.

### 8.3 Area-meaning helpers: `area_unit_is_sqyd` and `has_land_area_unit`

**What:**
- `area_unit_is_sqyd` = 1 if the advertiser gave the covered area in **square yards**;
- `has_land_area_unit` = 1 if the advert has a separate **land-area unit**.

**Why:** notebook 08 showed that "area" can mean the **built-up area** (flats) or the **plot area** (many houses, especially
when given in square yards). Houses stated in Sq-yrd have a much smaller area at a similar price. These two flags give
the model a hint that the area may be a plot area. They do not change the area itself.

**Fixed or learned?** Fixed: they come from the listing's own form fields.

In [14]:
# make_area_meaning_flags() is defined in src/features/build_features.py
train_sqyd, train_land_unit = make_area_meaning_flags(train)
print("area_unit_is_sqyd = 1 by property type:")
print(train_sqyd.groupby(train["property_type"]).sum().to_string())
print()
print("has_land_area_unit = 1 by property type:")
print(train_land_unit.groupby(train["property_type"]).sum().to_string())

area_unit_is_sqyd = 1 by property type:
property_type
Apartment                    19
Builder Floor Apartment      11
Penthouse                     0
Residential House          1266
Villa                       236

has_land_area_unit = 1 by property type:
property_type
Apartment                    7
Builder Floor Apartment     26
Penthouse                    0
Residential House          697
Villa                      594


### 8.4 Floor information (apartment-like types only)

**What:**
- `apartment_floor`: the floor of the flat, with `Ground` = 0, `Upper Basement` = −1, `Lower Basement` = −2 (the same
  mapping as rule R3 in notebook 14);
- `apartment_building_floors`: `total_floors`, i.e. the height of the building.

Both are filled **only for apartment-like types** (Apartment, Builder Floor Apartment, Penthouse). For Residential
Houses and Villas they are set to missing ("not applicable").

**Why only apartments:** for a flat, "5th floor of a 20-floor building" has a clear meaning. For a house, `total_floors`
might mean the number of storeys, and `floor` has no clear meaning. Villas never have a floor. We decided **not** to
force one interpretation across different property types (decision 5). The model can still tell houses apart, because
`property_type` is a feature.

`floor_ratio` is **not** created (notebook 17: very weak relation with price).

**Fixed or learned?** Fixed. The missing values are filled in section 10 (learned).

In [15]:
# make_floor_features() is defined in src/features/build_features.py
train_floor, train_building_floors = make_floor_features(train)

example_rows = [train.index[(train["property_type"] == "Apartment") & (train["floor"] == "Ground")][0],
                train.index[(train["property_type"] == "Apartment") & (train["floor"] == "12")][0],
                train.index[(train["property_type"] == "Residential House") & (train["floor"] == "1")][0]]
pd.DataFrame({
    "property_type": train.loc[example_rows, "property_type"],
    "floor (before)": train.loc[example_rows, "floor"],
    "total_floors (before)": train.loc[example_rows, "total_floors"],
    "apartment_floor": train_floor[example_rows],
    "apartment_building_floors": train_building_floors[example_rows],
})

,property_type,floor (before),total_floors (before),apartment_floor,apartment_building_floors
32,Apartment,Ground,5.0,0.0,5.0
2135,Apartment,12,14.0,12.0,14.0
121,Residential House,1,1.0,NaN,NaN


## 9. Categorical feature engineering (fixed rules)

Every categorical feature gets its missing values replaced by the text `"unknown"` here. That is a fixed rule: "unknown"
is a real, visible category, and the model can learn that "not stated" behaves differently. For furnishing, for example,
the unknown group is much cheaper (notebook 17). The conversion into 0/1 columns (one-hot encoding) is a learned step
and happens in section 12.

### 9.1 Balconies → `balconies_band`

**What:** `1`, `2`, `3`, `4+` (4 or more, including `"> 10"`), and `unknown`.

**Why missing is kept separate (not 0):** the website **never records 0 balconies**. A missing value therefore mixes
"no balcony" and "not stated", and we cannot tell which. Writing 0 or a median would invent a fact. The large values
(5, 6, … > 10) are rare, so they share one band `4+`.

In [16]:
# make_balconies_band() is defined in src/features/build_features.py
train_balconies_band = make_balconies_band(train["balconies"])

# Every original value must have landed in a band (no value silently treated as unknown)
unexpected = train["balconies"].notna() & (train_balconies_band == "unknown")
assert unexpected.sum() == 0, train.loc[unexpected, "balconies"].unique()

pd.crosstab(train["balconies"].fillna("(missing)"), train_balconies_band, colnames=["balconies_band"])

balconies_band,1,2,3,4+,unknown
balconies,,,,,
(missing),0,0,0,0,4990
1,4018,0,0,0,0
10,0,0,0,13,0
2,0,4692,0,0,0
3,0,0,1736,0,0
4,0,0,0,506,0
5,0,0,0,105,0
6,0,0,0,78,0
7,0,0,0,6,0


### 9.2 Transaction type → `transaction_group`

**What:** three groups: `Resale`, `New Property`, `other/unknown` (= `Other`, `Rent` and missing).

**Why:** every listing is a **sale** listing. Notebook 05 found that the website's `Rent` label is a data-entry mistake
(sale adverts entered on the rental form), so it says nothing about the property. `Other` and missing are tiny groups.
`transaction_type` is **not** used as a sale/rent filter; no row is removed.

In [17]:
# make_transaction_group() is defined in src/features/build_features.py
train_transaction_group = make_transaction_group(train["transaction_type"])
pd.crosstab(train["transaction_type"].fillna("(missing)"), train_transaction_group, colnames=["transaction_group"])

transaction_group,New Property,Resale,other/unknown
transaction_type,,,
(missing),0,0,29
New Property,2242,0,0
Other,0,0,44
Rent,0,0,12
Resale,0,13892,0


### 9.3 Possession → `possession_status`

**What:** `ready`, `under construction` or `unknown`, with the rule from notebook 17 (section 7.2):

| `possession` | status |
|---|---|
| Ready to Move, Immediately | `ready` |
| Under Construction, Ongoing, a future month such as `Dec '26` | `under construction` |
| Select Date, missing | `unknown` |

**Why:** the raw column has about 50 different values, mostly individual months. What matters for the price is whether
the home is ready. All month values lie in the future (Oct 2026 – Jan 2032), so they mean "not ready yet".
"Months until possession" is **not** created: it would be a date feature (notebook 17).

In [18]:
# possession_to_status() is defined in src/features/build_features.py (it handles ONE value, so we use apply)
train_possession_status = train["possession"].apply(possession_to_status)

examples = train[["possession"]].copy()
examples["possession_status"] = train_possession_status
print(examples.drop_duplicates().head(8).to_string(index=False))
print()
print(train_possession_status.value_counts().to_string())

        possession  possession_status
           Dec '28 under construction
           Feb '27 under construction
     Ready to Move              ready
           Aug '27 under construction
Under Construction under construction
               NaN            unknown
       Immediately              ready
           Dec '26 under construction

possession
ready                 14406
under construction     1493
unknown                 320


### 9.4 Facing → `facing_clean`

**What:** the eight compass directions stay as they are. Everything else becomes `unknown`: missing values,
"Newly Constructed Property" and landmark text such as "Near Income Tax Tower".

**Why:** those other values are not directions (notebook 17, section 8.1). Treating each of them as a separate
"direction" would create meaningless one-listing categories.

In [19]:
# make_facing_clean() is defined in src/features/build_features.py
train_facing = make_facing_clean(train["facing"])
print(pd.DataFrame({"before": train["facing"].fillna("(missing)"), "after": train_facing})
      .value_counts().head(12).to_string())

before                        after       
(missing)                     unknown         6411
East                          East            4597
West                          West            2745
North                         North            991
North - East                  North - East     645
South                         South            306
Newly Constructed Property    unknown          189
North - West                  North - West     145
South - East                  South - East     108
South -West                   South -West       59
Near Usha Mullapadi Hospital  unknown            1
Near Income Tax Tower         unknown            1


### 9.5 Other categorical columns

- `property_type`, `posted_by`: never missing, used as they are.
- `furnishing`, `age_of_property`: missing → `unknown`. For `age_of_property`, under-construction homes are usually
  `unknown`. That needs no extra category, because `possession_status` already tells the model the home is under construction.

In [20]:
for column in ["property_type", "posted_by", "furnishing", "age_of_property"]:
    print(train[column].fillna("unknown").value_counts().to_string())
    print()

property_type
Apartment                  9218
Residential House          4353
Villa                      2116
Builder Floor Apartment     498
Penthouse                    34

posted_by
Owner      14714
Agent       1275
Builder      230

furnishing
Unfurnished       7931
Semi-Furnished    4951
Furnished         3009
unknown            328

age_of_property
unknown              5891
New Construction     3583
Less than 5 years    2230
5 to 10 years        2041
10 to 15 years       1221
Above 20 years        653
15 to 20 years        600



### 9.6 Binary flags: `prime_location_flag` and `in_named_project`

**`prime_location_flag`** = 1 if the website's `prime_location` field is `Y`, else 0. It is a **listing-level attribute
provided by the website**. Its meaning is **not documented**, and it does **not** show that a locality is
geographically "prime": in notebook 17, 819 of 1,830 localities had both `Y` and `N` listings, and no price threshold was
found. It is available for any new advert, so it can be used as it is.

**`in_named_project`** = 1 if the listing has a project ID, else 0. The project ID is the website's `psmid` field from
`raw_json`, which notebook 15 extracted as `project_id`. (`project_name` is **not** used: it holds the website's
generated listing title, not the project.)

**Why `in_named_project` may be useful:** a listing linked to a named project is usually part of a builder's project or
a gated society (64% of apartments, 6% of houses). Such homes often come with shared facilities and a known builder,
which may affect the asking price. Only the yes/no flag is used. The ID itself would let the model memorise individual
projects.

**Fixed or learned?** Both are fixed rules.

In [21]:
# make_binary_flags() is defined in src/features/build_features.py
assert set(train["prime_location"].unique()) == {"Y", "N"}, "prime_location has unexpected values"
train_prime, train_in_project = make_binary_flags(train)

example_rows = [train.index[train["project_id"].notna()][0], train.index[train["project_id"].isna()][0]]
pd.DataFrame({
    "prime_location (before)": train.loc[example_rows, "prime_location"],
    "prime_location_flag": train_prime[example_rows],
    "project_id (before)": train.loc[example_rows, "project_id"],
    "in_named_project": train_in_project[example_rows],
})

,prime_location (before),prime_location_flag,project_id (before),in_named_project
0,Y,1,5186594.0,1
6,Y,1,NaN,0


## 10. Missing-value handling

**What:** after the fixed rules, missing values remain **only in numeric columns**. Every categorical column has an
explicit `unknown`. The numeric gaps are filled with the **median of the training data** by scikit-learn's
`SimpleImputer(strategy="median")` inside the preprocessing (section 12).

**Why the median:** it is not pulled up by the few huge houses or very tall buildings, unlike the mean.

**Learned from training data only:** the imputer **learns** one median per column when it is fitted on `train`
(section 13), and fills **both** train and test with those same training medians. The test set's own values never
affect the fill value.

| Column | Missing because | Fill value (learned from train) | Extra indicator? |
|---|---|---|---|
| `log_area` | no area given, or unreliable area (R2) | training median of `log_area` | **yes**: `area_missing` (section 8.2) |
| `bathrooms` | not stated (0.5%) | training median | **no** (see below) |
| `apartment_floor`, `apartment_building_floors` | not applicable (houses, villas) or not stated (≈ 5% of flats) | training median (calculated from the apartment-like listings, the only ones with values) | no: `property_type` shows which listings are houses/villas |
| `locality_centre_lat`, `locality_centre_long` | unknown locality, ID not in the table, or centre outside Hyderabad | training median | no: the `unknown locality` category marks most of these rows |

**Bathrooms: why no missing indicator?** Only 88 training listings (0.5%) have no bathroom count, so a separate column
would be almost always 0. Notebook 17 proposed no indicator for the same reason. The check below compares their
median asking price with the rest (training data, descriptive only).

In [22]:
bathrooms_example = rooms_to_number(train["bathrooms"])
bathroom_missing = bathrooms_example.isna()

print("Training listings with missing bathrooms:", bathroom_missing.sum(), f"({bathroom_missing.mean():.1%})")
print("Median asking price, bathrooms missing: ₹", round(y_train[bathroom_missing].median() / 1e5, 1), "lakh")
print("Median asking price, bathrooms given:   ₹", round(y_train[~bathroom_missing].median() / 1e5, 1), "lakh")

Training listings with missing bathrooms: 88 (0.5%)
Median asking price, bathrooms missing: ₹ 167.5 lakh
Median asking price, bathrooms given:   ₹ 150.0 lakh


**Result:** the 88 listings have a somewhat higher median (₹167.5 L vs ₹150 L). With so few listings this difference
is noisy, and the model already sees their bedrooms, area and type. We therefore start **without** a
`bathrooms_missing` indicator. This is a judgement call: it is easy to add (`SimpleImputer(add_indicator=True)`) if
notebook 19 shows it helps in cross-validation.

## 11. Geographic features: locality centre

**What:** two numeric features, `locality_centre_lat` and `locality_centre_long`: the centre of the listing's locality.

**Why the locality centre, and not the listing's own map pin:** notebook 17 (section 6.4) found that listing pins are
missing for 37% of listings, and 42% of the rest are just a copy of the locality centre. Some pins are far outside
Hyderabad. The locality centre is one fixed value per locality, and it also gives **rare** localities a position. A
model can use it to learn that nearby localities have similar prices.

**Three fixed rules:**
1. Matched listings already have the centre (`loc_locality_lat` / `loc_locality_long`).
2. Listings recovered in section 7.1 get the centre of their recovered ID from the website's locality table
   (`localities.csv`). We first check that this table gives exactly the same centre as the cleaned file for listings
   that already have one.
3. A centre **outside the Hyderabad box** used in notebook 14 (latitude 16.8–18.2, longitude 77.8–79.2) is set to
   missing. Some of these are clearly wrong (e.g. Gopanpally, a Hyderabad locality, at longitude 77.52). The locality ID
   itself is kept.

We never move, swap or guess coordinates. Missing centres are filled with the training median in section 12 (learned).

**Leakage risk:** none. Coordinates come from the locality table, not from prices, and no website price statistic is used.

In [23]:
# make_centre_lookups() is defined in src/features/build_features.py:
# two dictionaries, locality_id -> centre latitude and locality_id -> centre longitude
centre_lat_by_id, centre_long_by_id = make_centre_lookups(locality_table)

# Check: for listings that already have a centre, the locality table gives exactly the same values
has_centre = train["loc_locality_lat"].notna()
looked_up_lat = train.loc[has_centre, "locality_id"].map(centre_lat_by_id)
looked_up_long = train.loc[has_centre, "locality_id"].map(centre_long_by_id)
assert np.allclose(looked_up_lat, train.loc[has_centre, "loc_locality_lat"])
assert np.allclose(looked_up_long, train.loc[has_centre, "loc_locality_long"])
print("The locality table reproduces the existing centre for all", has_centre.sum(), "matched training listings.")

The locality table reproduces the existing centre for all 15016 matched training listings.


In [24]:
# make_locality_centre() is defined in src/features/build_features.py.
# The Hyderabad box (16.8-18.2, 77.8-79.2) is stored there as HYDERABAD_LAT_RANGE / HYDERABAD_LONG_RANGE.
train_centre_lat, train_centre_long = make_locality_centre(train, train_locality_id,
                                                           centre_lat_by_id, centre_long_by_id)

outside_box = train["loc_locality_lat"].notna() & train_centre_lat.isna()
print("Recovered listings that received a centre:", (recovered & train_centre_lat.notna()).sum(), "of", recovered.sum())
print("Centres set to missing (outside the box): ", outside_box.sum(), "listings in",
      train.loc[outside_box, "locality_id"].nunique(), "localities")
print("Training listings without a usable centre:", train_centre_lat.isna().sum())
print()
print("Localities whose centre was set to missing:")
outside_localities = train.loc[outside_box, ["locality_id", "loc_locality_lat", "loc_locality_long"]].drop_duplicates()
outside_localities["name"] = outside_localities["locality_id"].map(name_by_id)
outside_localities

Recovered listings that received a centre: 85 of 85
Centres set to missing (outside the box):  20 listings in 9 localities
Training listings without a usable centre: 1138

Localities whose centre was set to missing:


,locality_id,loc_locality_lat,loc_locality_long,name
121,165794.0,17.361101,79.215744,Suraram
238,135915.0,17.676638,77.616127,Zaheerabad
2631,114536.0,18.873041,79.446022,Mancherial
8327,163946.0,17.677628,77.615906,Ngos Colony
10413,163972.0,18.127581,79.205101,Other
10836,163084.0,17.673424,77.515541,Gopanpally
11014,164917.0,17.346258,79.207848,Bachuppala
16034,163138.0,18.300999,79.105423,Gundlapalli
16346,164130.0,17.600424,77.768791,Pothireddypalle


## 12. Build the preprocessing

### 12.1 One function for all fixed rules

**What:** `add_fixed_features(table, locality_table)` (in `build_features.py`) collects the fixed rules of sections
7–11 in one place and returns a new table with one column per baseline feature. It reads **only** `INPUT_COLUMNS` from
the listings, and only the locality centres from the website's locality table (section 11).

**Why one function:** train and test must receive **exactly** the same fixed rules. Calling the same function on both
guarantees that. The function learns nothing, so calling it on the test set is safe.

In [25]:
# add_fixed_features() is defined in src/features/build_features.py
train_features = add_fixed_features(train, locality_table)
test_features = add_fixed_features(test, locality_table)

print("Fixed-rule features, train:", train_features.shape)
print("Fixed-rule features, test: ", test_features.shape)
train_features.head()

Fixed-rule features, train: (16219, 21)
Fixed-rule features, test:  (4081, 21)


,bedrooms,bathrooms,log_area,area_missing,area_unit_is_sqyd,has_land_area_unit,apartment_floor,apartment_building_floors,locality_code,locality_centre_lat,locality_centre_long,property_type,balconies_band,transaction_group,furnishing,possession_status,age_of_property,facing_clean,posted_by,prime_location_flag,in_named_project
0,1,NaN,6.678342,0,0,0,NaN,NaN,106441,17.513124,78.259697,Apartment,unknown,New Property,Unfurnished,under construction,unknown,unknown,Agent,1,1
1,2,2.0,6.814543,0,0,0,NaN,NaN,106441,17.513124,78.259697,Apartment,unknown,New Property,Unfurnished,under construction,unknown,unknown,Agent,1,1
2,2,2.0,6.583409,0,0,0,NaN,NaN,93715,17.516449,78.596863,Apartment,unknown,New Property,Unfurnished,ready,unknown,unknown,Builder,1,1
3,2,2.0,6.698268,0,0,0,1.0,1.0,79094,17.505787,78.565155,Apartment,1,Resale,Unfurnished,ready,New Construction,West,Agent,1,1
4,2,2.0,7.030857,0,0,0,NaN,NaN,127553,17.458002,78.131844,Apartment,2,New Property,Unfurnished,under construction,Less than 5 years,unknown,Agent,1,1


### 12.2 The learned preprocessing (`ColumnTransformer`)

**What:** a `ColumnTransformer` applies a different step to each group of columns:

| Group | Columns | Step | What it learns from the training data |
|---|---|---|---|
| numeric | bedrooms, bathrooms, log_area, apartment floor ×2, locality centre ×2 | `SimpleImputer(strategy="median")` | one median per column |
| binary | area_missing, area_unit_is_sqyd, has_land_area_unit, prime_location_flag, in_named_project | `"passthrough"` (unchanged) | nothing (already 0/1, never missing) |
| categorical | property_type, balconies_band, transaction_group, furnishing, possession_status, age_of_property, facing_clean, posted_by | `OneHotEncoder(handle_unknown="ignore")` | the list of categories of each column |
| locality | locality_code | `OneHotEncoder(min_frequency=10, handle_unknown="infrequent_if_exist")` | which localities have ≥ 10 training listings |

**One-hot encoding** turns one text column into several 0/1 columns, one per category. For example, `furnishing` =
`"Semi-Furnished"` becomes `furnishing_Furnished` = 0, `furnishing_Semi-Furnished` = 1, `furnishing_Unfurnished` = 0,
`furnishing_unknown` = 0. Models need numbers, and one-hot encoding does this without pretending the categories have an order.

`handle_unknown="ignore"`: if the test set contains a category that never appeared in training, all its 0/1 columns
become 0 instead of raising an error.

**No scaling:** tree models do not need it. If notebook 19 uses a linear model, it adds a `StandardScaler` inside its
own model pipeline (decision 14).

`set_output(transform="pandas")` makes the result a DataFrame with column names, so we can inspect it.

In [26]:
NUMERIC_FEATURES = ["bedrooms", "bathrooms", "log_area", "apartment_floor", "apartment_building_floors",
                    "locality_centre_lat", "locality_centre_long"]
BINARY_FEATURES = ["area_missing", "area_unit_is_sqyd", "has_land_area_unit", "prime_location_flag", "in_named_project"]
CATEGORICAL_FEATURES = ["property_type", "balconies_band", "transaction_group", "furnishing", "possession_status",
                        "age_of_property", "facing_clean", "posted_by"]
LOCALITY_FEATURES = ["locality_code"]

all_feature_inputs = NUMERIC_FEATURES + BINARY_FEATURES + CATEGORICAL_FEATURES + LOCALITY_FEATURES
assert sorted(all_feature_inputs) == sorted(train_features.columns), "every built column must be in exactly one group"

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", SimpleImputer(strategy="median"), NUMERIC_FEATURES),
        ("binary", "passthrough", BINARY_FEATURES),
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_FEATURES),
        ("locality", OneHotEncoder(min_frequency=MIN_LOCALITY_LISTINGS, handle_unknown="infrequent_if_exist",
                                   sparse_output=False), LOCALITY_FEATURES),
    ]
)
preprocessor.set_output(transform="pandas")
preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('binary', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``fe

## 13. Fit the preprocessing on the training data only

**What:** `fit` is the moment the preprocessing **learns**: the medians, the category lists and the frequent-locality
list. We call it with **`train_features` only**.

**Leakage risk:** this is the step where leakage would happen if the test set were included. It is not.

**Note for notebook 19:** during cross-validation, this preprocessing must be placed **inside** a `Pipeline` with the
model, so it is re-fitted on each training fold. The object fitted here is for inspecting and checking the features.

In [27]:
preprocessor.fit(train_features)   # <- training data only

numeric_imputer = preprocessor.named_transformers_["numeric"]
locality_encoder = preprocessor.named_transformers_["locality"]

print("Medians learned from the training data:")
print(pd.Series(numeric_imputer.statistics_, index=NUMERIC_FEATURES).round(3).to_string())
print()
print("Locality categories seen in training:    ", len(locality_encoder.categories_[0]))
print("... of which grouped as rare (infrequent):", len(locality_encoder.infrequent_categories_[0]))

Medians learned from the training data:
bedrooms                      3.000
bathrooms                     3.000
log_area                      7.468
apartment_floor               4.000
apartment_building_floors     6.000
locality_centre_lat          17.447
locality_centre_long         78.409

Locality categories seen in training:     1835
... of which grouped as rare (infrequent): 1605


## 14. Transform train and test

**What:** `transform` applies what was learned. It **does not learn anything new**: the test set is filled with the
**training** medians, encoded with the **training** category lists, and its localities are checked against the
**training** frequent-locality list.

In [28]:
X_train = preprocessor.transform(train_features)
X_test = preprocessor.transform(test_features)

print("X_train:", X_train.shape)
print("X_test: ", X_test.shape)

X_train: (16219, 282)
X_test:  (4081, 282)


## 15. Inspect the resulting feature matrix

### 15.1 Columns per group

scikit-learn names each output column `<group>__<column>`, for example `numeric__log_area` or
`categorical__furnishing_unknown`.

In [29]:
group_of_column = pd.Series(X_train.columns).str.split("__").str[0]
print(group_of_column.value_counts().to_string())
print()
print("Total features:", X_train.shape[1])

locality       231
categorical     39
numeric          7
binary           5

Total features: 282


### 15.2 Example: numeric imputation

A training listing with **missing bathrooms** and one with a **missing area**, before and after the imputer. The filled
value is the training median, and `area_missing` = 1 keeps the fact that the area was unknown.

In [30]:
bathroom_row = train_features.index[train_features["bathrooms"].isna()][0]
area_row = train_features.index[train_features["log_area"].isna()][0]

for row, columns in [(bathroom_row, ["bathrooms"]), (area_row, ["log_area", "area_missing"])]:
    print("Listing", train.loc[row, "property_id"])
    for column in columns:
        group = "numeric" if column in NUMERIC_FEATURES else "binary"
        print(f"  {column:13} before: {train_features.loc[row, column]!s:6}  "
              f"after: {X_train.loc[row, group + '__' + column]}")
    print()

Listing listing_001
  bathrooms     before: nan     after: 3.0

Listing listing_002
  log_area      before: nan     after: 7.467942332285852
  area_missing  before: 1       after: 1



### 15.3 Example: categorical encoding

One listing's `furnishing` and `balconies_band`, and the 0/1 columns they became.

In [31]:
row = train_features.index[(train_features["balconies_band"] == "unknown") &
                           (train_features["furnishing"] == "Semi-Furnished")][0]

print("Before: furnishing =", repr(train_features.loc[row, "furnishing"]),
      "  balconies_band =", repr(train_features.loc[row, "balconies_band"]))
print("After:")
encoded_columns = [column for column in X_train.columns
                   if column.startswith("categorical__furnishing") or column.startswith("categorical__balconies_band")]
print(X_train.loc[row, encoded_columns].to_string())

Before: furnishing = 'Semi-Furnished'   balconies_band = 'unknown'
After:
categorical__balconies_band_1             0.0
categorical__balconies_band_2             0.0
categorical__balconies_band_3             0.0
categorical__balconies_band_4+            0.0
categorical__balconies_band_unknown       1.0
categorical__furnishing_Furnished         0.0
categorical__furnishing_Semi-Furnished    1.0
categorical__furnishing_Unfurnished       0.0
categorical__furnishing_unknown           0.0


### 15.4 Example: locality encoding (frequent, rare and unknown)

For each case: the listing's locality before, and which locality column is 1 afterwards.

In [32]:
locality_columns = [column for column in X_train.columns if column.startswith("locality__")]


def active_locality_column(row):
    # The one locality column that is 1 for this listing
    values = X_train.loc[row, locality_columns]
    return values[values == 1].index.tolist()


example_rows = {
    "frequent locality": train_features.index[train_features["locality_code"] == most_common_id][0],
    "rare locality": train_features.index[train_features["locality_code"] == rare_id][0],
    "unknown locality": train_features.index[train_features["locality_code"] == "unknown locality"][0],
}
for case, row in example_rows.items():
    code_value = train_features.loc[row, "locality_code"]
    name = name_by_id.get(int(code_value), "-") if code_value != "unknown locality" else "-"
    print(f"{case:18} before: {code_value!r:20} ({name})")
    print(f"{'':18} after:  {active_locality_column(row)}")

frequent locality  before: '84461'              (Kondapur)
                   after:  ['locality__locality_code_84461']
rare locality      before: '93664'              (Bongloor)
                   after:  ['locality__locality_code_infrequent_sklearn']
unknown locality   before: 'unknown locality'   (-)
                   after:  ['locality__locality_code_unknown locality']


### 15.5 First rows of the final training matrix

In [33]:
X_train.iloc[:5, :25]

,numeric__bedrooms,numeric__bathrooms,numeric__log_area,numeric__apartment_floor,numeric__apartment_building_floors,numeric__locality_centre_lat,numeric__locality_centre_long,binary__area_missing,binary__area_unit_is_sqyd,binary__has_land_area_unit,binary__prime_location_flag,binary__in_named_project,categorical__property_type_Apartment,categorical__property_type_Builder Floor Apartment,categorical__property_type_Penthouse,categorical__property_type_Residential House,categorical__property_type_Villa,categorical__balconies_band_1,categorical__balconies_band_2,categorical__balconies_band_3,categorical__balconies_band_4+,categorical__balconies_band_unknown,categorical__transaction_group_New Property,categorical__transaction_group_Resale,categorical__transaction_group_other/unknown
0,1.0,3.0,6.678342,4.0,6.0,17.513124,78.259697,0,0,0,1,1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
1,2.0,2.0,6.814543,4.0,6.0,17.513124,78.259697,0,0,0,1,1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
2,2.0,2.0,6.583409,4.0,6.0,17.516449,78.596863,0,0,0,1,1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
3,2.0,2.0,6.698268,1.0,1.0,17.505787,78.565155,0,0,0,1,1,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,2.0,2.0,7.030857,4.0,6.0,17.458002,78.131844,0,0,0,1,1,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0


## 16. Leakage and sanity checks

Each check is an `assert`, so the notebook stops if something is wrong.

1. No forbidden column (price, identifiers, dates, website price statistics, ...) appears in any feature name.
2. No missing values remain in `X_train` or `X_test`.
3. Train and test have exactly the same columns, in the same order.
4. The imputer's medians are exactly the medians of the **training** features (the fit used training data).
5. The encoder's rare-locality list is exactly the list computed from **training** counts in section 7.3.
6. The target is not in the features, and `y` lines up with `X` row by row.
7. Binary columns contain only 0 and 1.

In [34]:
# 1. No forbidden column name inside any feature name
for forbidden in FORBIDDEN_COLUMNS:
    assert not any(forbidden in column for column in X_train.columns), forbidden
print("1. No forbidden column in the features.")

# 2. No missing values
assert X_train.isna().sum().sum() == 0 and X_test.isna().sum().sum() == 0
print("2. No missing values in X_train or X_test.")

# 3. Same columns
assert list(X_train.columns) == list(X_test.columns)
print("3. Train and test have the same", X_train.shape[1], "columns.")

# 4. Medians come from the training data
training_medians = train_features[NUMERIC_FEATURES].median()
assert np.allclose(numeric_imputer.statistics_, training_medians.values)
print("4. Imputer medians = medians of the training features.")

# 5. Rare localities come from the training counts
assert set(locality_encoder.infrequent_categories_[0]) == set(rare_localities)
print("5. Encoder's rare localities = the", len(rare_localities), "localities with < 10 training listings.")

# 6. Target separate and aligned
assert "price" not in train_features.columns
assert (X_train.index == y_train.index).all() and (X_test.index == y_test.index).all()
print("6. Target kept separate; X and y are aligned.")

# 7. Binary columns are 0/1
binary_columns = ["binary__" + column for column in BINARY_FEATURES]
assert X_train[binary_columns].isin([0, 1]).all().all() and X_test[binary_columns].isin([0, 1]).all().all()
print("7. Binary columns contain only 0 and 1.")

1. No forbidden column in the features.
2. No missing values in X_train or X_test.
3. Train and test have the same 282 columns.
4. Imputer medians = medians of the training features.
5. Encoder's rare localities = the 1605 localities with < 10 training listings.
6. Target kept separate; X and y are aligned.
7. Binary columns contain only 0 and 1.


In [35]:
# The source files are unchanged
for path, md5 in md5_at_start.items():
    assert file_md5(path) == md5, f"{path.name} changed!"
    print(f"{path.name:28} unchanged")

cleaned_listings.csv         unchanged


train_test_split.csv         unchanged
localities.csv               unchanged


## 17. Final feature list

In [36]:
feature_groups = pd.DataFrame({
    "feature (before encoding)": all_feature_inputs,
    "type": (["numeric"] * len(NUMERIC_FEATURES) + ["binary 0/1"] * len(BINARY_FEATURES)
             + ["categorical"] * len(CATEGORICAL_FEATURES) + ["categorical (rare grouping)"]),
})
feature_groups["columns after encoding"] = [
    sum(1 for column in X_train.columns if column.split("__")[1].startswith(feature))
    for feature in feature_groups["feature (before encoding)"]
]
feature_groups

,feature (before encoding),type,columns after encoding
0,bedrooms,numeric,1
1,bathrooms,numeric,1
2,log_area,numeric,1
3,apartment_floor,numeric,1
4,apartment_building_floors,numeric,1
5,locality_centre_lat,numeric,1
6,locality_centre_long,numeric,1
7,area_missing,binary 0/1,1
8,area_unit_is_sqyd,binary 0/1,1
9,has_land_area_unit,binary 0/1,1


In [37]:
print("Input training rows:        ", len(train))
print("Input test rows:            ", len(test))
print("Features before encoding:   ", len(all_feature_inputs))
print("Final features (columns):   ", X_train.shape[1])
print("  of which locality columns:", len(locality_columns))
print("    frequent real localities:", len(locality_columns) - 2)
print("    'unknown locality':       1")
print("    shared rare column:       1")

Input training rows:         16219
Input test rows:             4081
Features before encoding:    21
Final features (columns):    282
  of which locality columns: 231
    frequent real localities: 229
    'unknown locality':       1
    shared rare column:       1


## 18. Summary and what notebook 19 will do

### What was built

A feature matrix for the **baseline** model: **21 features before encoding, 282 columns after encoding** (7 numeric,
5 binary, 39 one-hot columns for 8 categorical features, 231 locality columns).

| Group | Features |
|---|---|
| Size | `bedrooms`, `bathrooms`, `log_area`, `area_missing` |
| Area meaning | `area_unit_is_sqyd`, `has_land_area_unit` |
| Building (apartment-like only) | `apartment_floor`, `apartment_building_floors` |
| Location | `locality_code` (frequent / rare / unknown), `locality_centre_lat`, `locality_centre_long` |
| Property | `property_type`, `balconies_band`, `furnishing`, `age_of_property`, `facing_clean`, `possession_status` |
| Listing | `transaction_group`, `posted_by`, `prime_location_flag`, `in_named_project` |

### Learned from the training data (fitted on `train` only)

- the medians used to fill missing numeric values (bathrooms, log area, apartment floor, building floors, locality centre);
- the category list of every one-hot-encoded column;
- the list of frequent localities (≥ 10 training listings); all others, and localities never seen in training, go to the
  rare column.

### Fixed rules (same for train and test, nothing learned)

`"> 10"` → 11; `log1p(area)`; `area_missing`; the Sq-yrd and land-unit flags; the floor mapping and "apartment-like
only"; the safe locality recovery; `"unknown locality"`; the centre lookup and the Hyderabad box; balcony bands;
transaction groups; possession status; the facing cleanup; missing → `"unknown"`; `prime_location_flag`;
`in_named_project`.

**The test set was not used to fit any preprocessing.** It was only transformed, and it had no part in any decision.

### Warnings and limitations

- **Asking price, not sale price.** The model will learn what sellers ask.
- **Area meaning differs by property type** (built-up vs plot). The flags help, but they cannot fix it per listing.
- **Imputed values are not real values.** Missing areas, floors and centres are filled with training medians. For
  houses and villas the apartment floor columns hold an "apartment median" that means "not applicable"; the model has to
  combine them with `property_type`.
- **Locality centres** are one point per locality. Nine localities (20 training listings) had centres outside Hyderabad
  and were set to missing. Listings in the same locality all get the same position.
- **`prime_location`** is an undocumented website attribute; the write-up should not describe it as "prime area".
- **Rare localities** (1,605 localities, 3,863 training listings) share one column, so the model cannot tell them apart
  except through the centre coordinates.
- **Bathrooms** have no missing indicator (88 training listings); see section 10.
- **The sampling is not random** (price-range searches). Errors must be reported per price range in notebook 19.
- **No data was saved.** Notebook 19 imports the fixed rules from `src/features/build_features.py` and defines the
  same learned preprocessing inside its model `Pipeline`.

### What notebook 19 will do

1. Rebuild these features with `add_fixed_features()` from `src/features/build_features.py`, and put the learned
   preprocessing inside a `Pipeline` together with the model, so it is fitted on each training fold only.
2. Decide whether to model `price` or `log(price)`.
3. Train a first baseline model and compare it with a very simple reference (e.g. predicting the training median),
   using `GroupKFold` cross-validation on the training set with `groups = property_group_id`.
4. Report errors overall, per price range and per property type. The test set is used only once, at the very end.